In [29]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

listings = pd.read_csv("/Users/seanshih/Desktop/IDX/CRMLSListed_All.csv")
listings = listings[listings["PropertyType"] == "Residential"]


/var/folders/5h/bxf_bvns78z4pt8s0drk3kcc0000gn/T/ipykernel_36081/23906899.py:5: DtypeWarning: Columns (2,82) have mixed types. Specify dtype option on import or set low_memory=False.
  listings = pd.read_csv("/Users/seanshih/Desktop/IDX/CRMLSListed_All.csv")


In [31]:
sold = pd.read_csv("/Users/seanshih/Desktop/IDX/CRMLS_Sold_All.csv")
sold = sold[sold["PropertyType"] == "Residential"]

/var/folders/5h/bxf_bvns78z4pt8s0drk3kcc0000gn/T/ipykernel_36081/3318648413.py:1: DtypeWarning: Columns (0,1,9,78,79,80,81,82) have mixed types. Specify dtype option on import or set low_memory=False.
  sold = pd.read_csv("/Users/seanshih/Desktop/IDX/CRMLS_Sold_All.csv")


In [33]:
listings.head

<bound method NDFrame.head of         OriginalListPrice  ListingKey                       ListAgentEmail  \
0                929000.0  1076194146                   dianne@drector.com   
1                999999.0  1076194026  realestateby_denisegarcia@gmail.com   
2               1400000.0  1076193814           alizabethjames@hotmail.com   
4                549000.0  1076193525                  parsanina@yahoo.com   
5                729999.0  1076193391            tony.labellarte@gmail.com   
...                   ...         ...                                  ...   
852949           615000.0  1077117033     kathleenwestwood@willisallen.com   
852950          2595000.0  1076449404      debbi.dimaggio@corcoranIcon.com   
852956           619000.0  1075400367       tanya@tanyajonessellshomes.com   
852960           409000.0  1058408504           marierealtor4you@gmail.com   
852962          1825000.0  1038248402               taylor@willisallen.com   

       CloseDate  ClosePrice List

In [35]:
listings.columns

Index(['OriginalListPrice', 'ListingKey', 'ListAgentEmail', 'CloseDate',
       'ClosePrice', 'ListAgentFirstName', 'ListAgentLastName', 'Latitude',
       'Longitude', 'UnparsedAddress', 'PropertyType', 'LivingArea',
       'ListPrice', 'DaysOnMarket', 'ListOfficeName', 'BuyerOfficeName',
       'CoListOfficeName', 'ListAgentFullName', 'CoListAgentFirstName',
       'CoListAgentLastName', 'BuyerAgentMlsId', 'BuyerAgentFirstName',
       'BuyerAgentLastName', 'FireplacesTotal', 'AssociationFeeFrequency',
       'AboveGradeFinishedArea', 'ListingKeyNumeric', 'MLSAreaMajor',
       'TaxAnnualAmount', 'CountyOrParish', 'PropertyType.1', 'MlsStatus',
       'ElementarySchool', 'ListAgentFirstName.1', 'AttachedGarageYN',
       'ParkingTotal', 'BuilderName', 'PropertySubType', 'LotSizeAcres',
       'SubdivisionName', 'BuyerOfficeAOR', 'YearBuilt', 'DaysOnMarket.1',
       'StreetNumberNumeric', 'LivingArea.1', 'ListingId',
       'BathroomsTotalInteger', 'City', 'TaxYear', 'BuildingAreaTot

In [37]:
#FRED

url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=MORTGAGE30US"
mortgage = pd.read_csv(url, parse_dates=['observation_date'])
mortgage.columns = ['date', 'rate_30yr_fixed']

In [39]:
mortgage['year_month'] = mortgage['date'].dt.to_period('M')
mortgage_monthly = (
mortgage.groupby('year_month')['rate_30yr_fixed']
.mean().reset_index()
)

In [41]:
# Sold dataset — key off CloseDate
sold['year_month'] = pd.to_datetime(sold['CloseDate']).dt.to_period('M')

# Listings dataset — key off ListingContractDate
listings['year_month'] = pd.to_datetime(
listings['ListingContractDate']).dt.to_period('M')

In [43]:
sold_with_rates = sold.merge(mortgage_monthly, on='year_month', how='left')
listings_with_rates = listings.merge(mortgage_monthly, on='year_month', how='left')

In [45]:
# Check for any unmatched rows (rate should not be null)
print(sold_with_rates['rate_30yr_fixed'].isnull().sum())
print(listings_with_rates['rate_30yr_fixed'].isnull().sum())
# Preview
print(sold_with_rates[['CloseDate', 'year_month', 'ClosePrice',
'rate_30yr_fixed']].head())

0
0
    CloseDate year_month  ClosePrice  rate_30yr_fixed
0  2024-09-12    2024-09   1090000.0             6.18
1  2024-09-30    2024-09   1995000.0             6.18
2  2024-09-30    2024-09   2340000.0             6.18
3  2024-09-30    2024-09    984000.0             6.18
4  2024-09-30    2024-09   1225000.0             6.18


In [ ]:
sold_with_rates.to_csv("/Users/seanshih/Desktop/IDX/CRMLS_Sold_Enriched.csv", index=False)

In [ ]:
listings_with_rates.to_csv("/Users/seanshih/Desktop/IDX/CRMLS_Listed_Enriched.csv", index=False)